# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmoud-mos/my-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

* Finding 1: "Content refreshes yield a 30% average organic traffic lift."

    Label Origin: The label relies on historical post-refresh impression/click changes.
    
    Methodology Question: How were pages selected for content refreshes? If high-performing or already-trending pages were selectively chosen for     refreshes, the observed traffic lift reflects selection bias rather than the causal impact of the refresh itself. Was a matched holdout     control group of unrefreshed pages evaluated across the same time window?


* Finding 2: "AI-generated meta titles improve Click-Through Rate (CTR) by 15% across domain verticals."

   Label Origin: Measured by comparing pre- and post-implementation CTR changes on modified URLs.
   
   Methodology Question: Does the validation design enforce entity isolation across client domains? If pages from the same domain/client are    split randomly across training and validation sets, the model can memorize site-specific baseline CTRs (data leakage) rather than learning    generalized title quality features.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import duckdb
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import GroupShuffleSplit

# 1. Load Data
load_dotenv()
hf_token = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

query = f"""
SELECT 
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    scroll_events,
    (gsc_sum_position / (gsc_impressions + 0.001)) as avg_position,
    CASE WHEN gsc_impressions > 200 AND gsc_clicks = 0 THEN 1 ELSE 0 END as target_needs_review
FROM read_parquet('{rel}', hive_partitioning=true)
WHERE gsc_data_available = TRUE;
"""
df = con.sql(query).df().fillna(0)

features = ['gsc_impressions', 'avg_position', 'scroll_events']
X = df[features]
y = df['target_needs_review']
groups = df['content_hash_id']

# 2. Week 5 Chronological Split (Baseline Split)
split_idx = int(len(df) * 0.8)
X_train_c, X_test_c = X.iloc[:split_idx], X.iloc[split_idx:]
y_train_c, y_test_c = y.iloc[:split_idx], y.iloc[split_idx:]

dt_c = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_c.fit(X_train_c, y_train_c)
preds_c = dt_c.predict(X_test_c)

# 3. Honest Grouped Split (Grouped by content_hash_id to prevent page-level leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

dt_g = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_g.fit(X_train_g, y_train_g)
preds_g = dt_g.predict(X_test_g)

# 4. Before / After Comparison
results = pd.DataFrame({
    'Validation Strategy': ['Chronological Split (Week 5)', 'Grouped Split (Week 6 - Honest)'],
    'Precision': [precision_score(y_test_c, preds_c), precision_score(y_test_g, preds_g)],
    'Recall': [recall_score(y_test_c, preds_c), recall_score(y_test_g, preds_g)],
    'F1 Score': [f1_score(y_test_c, preds_c), f1_score(y_test_g, preds_g)]
})

print(results.to_string(index=False))

            Validation Strategy  Precision   Recall  F1 Score
   Chronological Split (Week 5)   0.594325 0.804660  0.683681
Grouped Split (Week 6 - Honest)   0.562976 0.800318  0.660987


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# 1. Feature-Target Correlation Audit (Checking for direct proxy leakage)

correlations = df[features].apply(
    lambda col: col.corr(df['target_needs_review']))
print("--- Feature Correlation with Target ---")
print(correlations.to_string())

# 2. Inspect Failure Examples (False Positives and False Negatives under Grouped Split)
test_df_g = df.iloc[test_idx].copy()
test_df_g['pred'] = preds_g

false_positives = test_df_g[(
    test_df_g['target_needs_review'] == 0) & (test_df_g['pred'] == 1)]
false_negatives = test_df_g[(
    test_df_g['target_needs_review'] == 1) & (test_df_g['pred'] == 0)]

print("\n--- Top False Positives (Flagged as needing review, but actually OK) ---")
print(false_positives[['content_hash_id', 'gsc_impressions', 'gsc_clicks',
      'avg_position', 'scroll_events']].head(3).to_string(index=False))

print("\n--- Top False Negatives (Missed broken pages) ---")
print(false_negatives[['content_hash_id', 'gsc_impressions', 'gsc_clicks',
      'avg_position', 'scroll_events']].head(3).to_string(index=False))

--- Feature Correlation with Target ---
gsc_impressions    0.288091
avg_position      -0.018185
scroll_events      0.027200

--- Top False Positives (Flagged as needing review, but actually OK) ---
         content_hash_id  gsc_impressions  gsc_clicks  avg_position  scroll_events
content_aa63abfffe1cdaab              368           3      4.586944              0
content_b41b31b9512ca5fa              207           1      3.951672              0
content_2c842749a52ea6f4              327           1      1.752288              0

--- Top False Negatives (Missed broken pages) ---
         content_hash_id  gsc_impressions  gsc_clicks  avg_position  scroll_events
content_a15138d47e9949ae              531           0      0.802258              0
content_96d1914e4d53283f              721           0      4.486818              0
content_0f535bdc35ab3b87              571           0      6.744296              0


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

* Original Week 5 Claim:
"The Decision Tree model accurately detects low-engagement pages and proves that machine learning outperforms rule-based heuristics across all content performance metrics."

* Rewritten Safe Claim:
"Under an entity-grouped validation split (content_hash_id), the decision tree model demonstrated an observed F1 score of 0.66. Impression volume (gsc_impressions) drove over 90% of feature importance, indicating that the model serves as a directional decision-support heuristic for flagging potential review candidates rather than an automated reviewer."

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.